# Continued Pre-training (CPT) on Ascend NPU

This notebook runs **Continued Pre-training** — plain next-token prediction over a raw-text corpus — on a Huawei Ascend NPU (e.g. `Ascend910B4`) using only mainline `transformers` + `torch_npu`. No CUDA, no `bitsandbytes`, no `training_hub`.

> Why a separate NPU notebook? `training_hub` (the library `cpt-comprehensive-tutorial.ipynb` uses) targets NVIDIA GPUs via `torch.cuda`, `bitsandbytes`, and `unsloth`. None of those run on Ascend. The recipe here reproduces the same *training* using `transformers.Trainer` on `torch_npu`, so the resulting checkpoint is a drop-in HF-format model just like the CUDA path produces.

**Runtime prerequisites**

- Ascend driver + CANN runtime installed on the node (matching the workbench image's CANN version).
- Workbench image: `PyTorch CANN` (Python 3.12, CANN 8.5.0, PyTorch 2.9.0, `torch_npu 2.9.0`) or equivalent.
- At least 1 NPU visible via the Kubernetes device plugin (`huawei.com/Ascend910B4` or your cluster's resource name).
- Persistent storage for the base model + checkpoints (`/opt/app-root/src/...`).

## When to use CPT

CPT keeps the original **causal-LM** objective and updates *all* weights, so it's the tool for **injecting domain knowledge** before instruction tuning. Typical pipeline:

```
raw corpus  →  CPT (this notebook)  →  SFT / LoRA  →  alignment
```

CPT catastrophically forgets general-domain skills if you run it too long on a narrow corpus. Mitigations: keep learning rate small (`1e-5`–`5e-6`), mix general-domain text into the corpus, and always follow up with SFT/OSFT.

## Step 1 — Environment check

Confirm `torch_npu` is importable and at least one NPU is visible. `torch_npu` monkey-patches `torch` on import so `torch.npu.*` becomes available.

In [ ]:
import os
# NPU allocator tuning — expandable segments avoid fragmentation on Ascend.
os.environ.setdefault("PYTORCH_NPU_ALLOC_CONF", "expandable_segments:True")

import torch
import torch_npu  # noqa: F401  — side-effect import registers torch.npu
import transformers

print("torch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("transformers:", transformers.__version__)
print("NPU available:", torch.npu.is_available())
print("NPU count:", torch.npu.device_count())
if torch.npu.is_available():
    torch.npu.set_device(0)
    print("device 0:", torch.npu.get_device_name(0))
    # bf16 support is standard on Ascend910B — quick sanity check.
    x = torch.randn(64, 64, dtype=torch.bfloat16, device="npu")
    y = (x @ x.T).float().mean().item()
    print(f"bf16 matmul ok, mean={y:.4f}")

## Step 2 — Parameters

Edit these to point at your model and corpus. Defaults use a small base and short training so the notebook completes as a smoke test in a few minutes on a single NPU.

In [ ]:
# --- model & corpus ---
MODEL_PATH = os.environ.get("HF_MODEL_DIR", "/opt/app-root/src/models/Qwen2.5-0.5B")
CORPUS_PATH = os.environ.get("CORPUS_PATH", "/opt/app-root/src/datasets/cpt/corpus.jsonl")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/opt/app-root/src/cpt-npu-output")

# --- training ---
BLOCK_SIZE = 512               # chunk length used to build training samples
NUM_EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACC_STEPS = 4             # effective batch = 2 * 4 = 8 tokens/step
LEARNING_RATE = 5e-6           # keep small — CPT is easy to overfit / forget
WARMUP_STEPS = 20
SAVE_STEPS = 200
LOGGING_STEPS = 10

# --- compute ---
USE_BF16 = True                # Ascend910B supports bf16 natively
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("model :", MODEL_PATH)
print("corpus:", CORPUS_PATH)
print("output:", OUTPUT_DIR)

## Step 3 — Data format

CPT expects **raw text**, not chat turns. Each line of the JSONL is one document under a `text` field:

```json
{"text": "Alauda AI is an MLOps platform that runs training and inference on Kubernetes."}
{"text": "Continued pre-training adapts a base model to a new domain using unlabeled text."}
```

The next cell falls back to a tiny **built-in synthetic corpus** if `CORPUS_PATH` doesn't exist, so the notebook can be run end-to-end without any dataset upload — useful as a smoke test.

In [ ]:
import json
from pathlib import Path

if not Path(CORPUS_PATH).exists():
    fallback = Path(OUTPUT_DIR) / "synthetic_corpus.jsonl"
    docs = [
        "Alauda AI is an MLOps platform that runs fine-tuning and inference on Kubernetes.",
        "Continued pre-training keeps the causal-LM objective and updates every weight.",
        "Huawei Ascend NPUs are programmed through the CANN runtime and torch_npu.",
        "A CPT run should be followed by SFT to restore instruction-following behaviour.",
        "transformers.Trainer works on torch_npu out of the box once torch_npu is imported.",
    ] * 20
    with fallback.open("w") as f:
        for d in docs:
            json.dump({"text": d}, f); f.write("\n")
    CORPUS_PATH = str(fallback)
    print(f"corpus not found; wrote synthetic fallback to {CORPUS_PATH}")
print("corpus:", CORPUS_PATH)

## Step 4 — Load model + tokenizer, tokenize corpus

Load the base model in `bf16` (Ascend-native), then tokenize the corpus and pack it into fixed-length chunks of `BLOCK_SIZE`. For CPT, `labels == input_ids` — every token contributes to the loss.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
    attn_implementation="eager",  # SDPA works on NPU; eager is the safe default across CANN versions
)
model.to("npu")
model.gradient_checkpointing_enable()

raw = load_dataset("json", data_files=CORPUS_PATH, split="train")
print("raw documents:", len(raw))

def tokenize(batch):
    return tokenizer(batch["text"], add_special_tokens=False)

tokenized = raw.map(tokenize, batched=True, remove_columns=raw.column_names)

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_len = (len(concatenated["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [t[i : i + BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    result["labels"] = [ids.copy() for ids in result["input_ids"]]
    return result

packed = tokenized.map(group_texts, batched=True)
print("packed chunks:", len(packed), "of", BLOCK_SIZE, "tokens each")

## Step 5 — Train

Use plain `Trainer` with `bf16=True`. **Do not** set `fp16=True` — Ascend prefers bf16, and mixing fp16 with older CANN builds triggers dynamic-shape recompilation.

The optimizer is `adamw_torch` (mainline PyTorch). Do not use `adamw_bnb_8bit` — bitsandbytes is not upstream on NPU (see the QLoRA-NPU notebook for how to bring in the community `bitsandbytes-npu` fork if you need it).

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=USE_BF16,
    fp16=False,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to=[],
    remove_unused_columns=False,
    dataloader_pin_memory=False,  # pinned memory is not supported on NPU
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=packed,
    data_collator=collator,
)

train_result = trainer.train()
print(train_result.metrics)

## Step 6 — Save the HF-format checkpoint

Save the tuned weights and tokenizer under `OUTPUT_DIR/hf_format` so downstream SFT / inference recipes can load it with `AutoModelForCausalLM.from_pretrained(...)`.

In [ ]:
hf_dir = os.path.join(OUTPUT_DIR, "hf_format")
trainer.save_model(hf_dir)
tokenizer.save_pretrained(hf_dir)
print("saved:", hf_dir)
print("contents:", sorted(os.listdir(hf_dir)))

## Step 7 — Quick inference smoke

Load the freshly-saved checkpoint back onto the NPU and generate a few tokens. This is a *sanity* check — the base model is small and the training corpus is tiny, so don't judge quality here.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(hf_dir)
mdl = AutoModelForCausalLM.from_pretrained(hf_dir, torch_dtype=torch.bfloat16).to("npu")
mdl.eval()

prompt = "Alauda AI is"
inputs = tok(prompt, return_tensors="pt").to("npu")
with torch.no_grad():
    out = mdl.generate(**inputs, max_new_tokens=32, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))

## Notes and gotchas on NPU

- **Import order.** `import torch_npu` **before** the first `torch.npu.*` call — it registers the backend.
- **Precision.** Use `bf16`. Avoid `fp16` unless you know your CANN build supports it end-to-end; mixing fp16 with older builds triggers dynamic-shape retracing that stalls training.
- **Optimizer.** Stick to `adamw_torch`. `adamw_bnb_8bit` requires `bitsandbytes` — see the [QLoRA-NPU tutorial](./qlora-npu-tutorial.ipynb) for how to install the Ascend fork if you need paged optimizers.
- **Attention backend.** `attn_implementation="eager"` is the portable default. `sdpa` works on most CANN 8.x builds but occasionally hits an unsupported-op path on prerelease combinations. `flash_attention_2` is CUDA-only.
- **Data loader.** Set `dataloader_pin_memory=False`; NPU does not use CUDA pinned host memory and the flag can trip guard checks.
- **Multi-NPU.** For >1 NPU on a single node, launch this notebook's script form under `torch.distributed` via `torchrun --nproc_per_node=<npu_count>`. `Trainer` picks up `torch.distributed` transparently on NPU thanks to `hccl`.
- **Catastrophic forgetting.** Keep `learning_rate` in the `1e-5` – `5e-6` range for CPT and always follow up with SFT — CPT alone breaks instruction-following.